Tensorflow and Keras were installed following the official tutorial:
https://www.tensorflow.org/install/pip

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
import logging
logger = tf.get_logger()
logger.setLevel(logging.ERROR)

from tensorflow import keras

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import shap

import sys
sys.path.append("..")
import methods.explainability as ex
from methods import utils

physical_devices = tf.config.list_physical_devices("GPU")
print("Num GPUs:", len(physical_devices))

In [ ]:
from skimage.filters.rank import entropy
from skimage.morphology import disk
from skimage.exposure import rescale_intensity
from skimage.measure import shannon_entropy
import skimage as ski
from scipy.ndimage import distance_transform_edt, binary_fill_holes

def get_nucleus_mask(img, res, tolerance=0.2):

    sigma = 0.1 / res

    smooth_img = ski.filters.gaussian(img, sigma=sigma)
    otsu_thres = ski.filters.threshold_otsu(smooth_img)
    lower_thres = otsu_thres * (1 - tolerance)
    upper_thres = otsu_thres * (1 + tolerance)

    # applies histerisis thresholding of TOLERANCE
    mask = ski.filters.apply_hysteresis_threshold(
        smooth_img, lower_thres, upper_thres
    )

    # fills holes
    mask = ski.morphology.binary_closing(mask, ski.morphology.ball(5))
    mask = binary_fill_holes(mask, ski.morphology.ball(5))

    return mask



def compute_exp_signal_by_distance(explanations, ims):

    data = []

    for exp_im, im in zip(explanations, ims):

        nuc_mask = get_nucleus_mask(im, RES)
        distance_map = distance_transform_edt(nuc_mask)  # Compute distance transform
    
        #plt.imshow(distance_map)
        #plt.show()
        #plt.imshow(nuc_mask)
        #plt.show()
        #plt.imshow(np.squeeze(im))
        #plt.show()
    
        # Flatten arrays
        distances = distance_map.flatten()
        exp_values = exp_im.flatten()
    
        # Bin distances and compute mean SHAP value per distance
        for d in np.unique(distances):
            mean_exp = exp_values[distances == d].mean()  # Compute mean SHAP for this distance
            data.append((int(d), mean_exp))

    df = pd.DataFrame(data, columns=["Distance", "EXP_Value"])
    return df


def get_explanation_entropy(explanations, disk_size):
    
    footprint = disk(disk_size)
    local_ent = []
    shannon_ent = []
    
    for ex_im in explanations:
    
        ex_im = np.squeeze(ex_im)
        ex_im = rescale_intensity(ex_im, in_range=(ex_im.min(), ex_im.max()), out_range=(0, 255))
        ex_im = ex_im.astype(np.uint8)

        ent = entropy(ex_im, footprint)
        local_ent.extend(ent.ravel())
        shannon_ent.append(shannon_entropy(ex_im))

        """
        plt.imshow(ex_im)
        plt.show()
        plt.imshow(ent)
        plt.show()
        """

    return local_ent, shannon_ent

In [ ]:
NOTEBOOK = "2D_keras_binary_L"
NORM = "none"
RES = 0.05
plane = "XY"
DIMS = "2D"
k_cv = 5
IMGS_DIR = f"../../data/preprocessed/2D_res={RES}_norm={NORM}_{k_cv}fold_withDMSO"

CNN_DENSE_FILTS = (256, 64, 16)
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 64
CHANNEL_MODE = "grayscale"
N_OUTPUT_UNITS = 1
LR_SCHED = None
OPTIMIZER = "AdamW"
COMMENT = "testing baseline"

EPOCHS = 50
LR = 1e-4
SEED = 2023

if CHANNEL_MODE == "rgb":
    channels = (3,)
elif CHANNEL_MODE == "grayscale":
    channels = (1,)

if N_OUTPUT_UNITS == 1:
    OUTPUT_FUNC = "sigmoid"
    LOSS_FUNC = tf.keras.losses.BinaryCrossentropy(
        label_smoothing=0.1,
    )
    LABEL_MODE = "binary"
elif N_OUTPUT_UNITS == 2:
    OUTPUT_FUNC = "softmax"
    LOSS_FUNC = "categorical_crossentropy"
    LABEL_MODE = "categorical"

In [ ]:
model = keras.models.load_model("../models/chromagenet/calmodel_fold_1.keras")
model.load_weights("../models/checkpoints/fold_1/44-0.692.weights.h5")

lr_obj = LR

if OPTIMIZER == "AdamW":
    optim = keras.optimizers.AdamW(
        learning_rate=lr_obj, use_ema=True
    )
elif OPTIMIZER == "Adam":
    optim = keras.optimizers.Adam(learning_rate=lr_obj, use_ema=True)
elif OPTIMIZER == "SGD":
    optim = keras.optimizers.SGD(learning_rate=lr_obj, momentum=0.6)

metrics = [
    "accuracy",
    tf.keras.metrics.AUC(),
    tf.keras.metrics.Precision(),
    tf.keras.metrics.Recall(),
    tf.keras.metrics.F1Score(),
]

model.compile(
    optimizer=optim,
    loss=LOSS_FUNC,
    metrics=metrics,
    jit_compile=True,
)

In [ ]:
k = 1 
train_ds = None

for i in range(1, k_cv + 1):
    if i == k:
        val_ds = tf.keras.utils.image_dataset_from_directory(
            f"{IMGS_DIR}/fold_{i}/",
            color_mode=CHANNEL_MODE,
            labels="inferred",
            label_mode=LABEL_MODE,
            interpolation="bilinear",
            seed=SEED,
            image_size=IMAGE_SIZE,
            batch_size=BATCH_SIZE,
            shuffle=False,
        )
        continue

    if train_ds:
        train_ds = train_ds.concatenate(
            tf.keras.utils.image_dataset_from_directory(
                f"{IMGS_DIR}/fold_{i}/",
                color_mode=CHANNEL_MODE,
                labels="inferred",
                label_mode=LABEL_MODE,
                interpolation="bilinear",
                seed=SEED,
                image_size=IMAGE_SIZE,
                batch_size=BATCH_SIZE,
                shuffle=True,
            )
        )
    else:
        train_ds = tf.keras.utils.image_dataset_from_directory(
            f"{IMGS_DIR}/fold_{i}/",
            color_mode=CHANNEL_MODE,
            labels="inferred",
            label_mode=LABEL_MODE,
            interpolation="bilinear",
            seed=SEED,
            image_size=IMAGE_SIZE,
            batch_size=BATCH_SIZE,
            shuffle=True,
        )

n_train_ims = train_ds.cardinality().numpy() * BATCH_SIZE
print(n_train_ims)

train_ds = train_ds.unbatch().shuffle(10000).batch(BATCH_SIZE)
# Prefetching samples in GPU memory helps maximize GPU utilization.
train_ds = train_ds.cache().prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.cache().prefetch(tf.data.AUTOTUNE)

In [ ]:
model.evaluate(val_ds)

In [ ]:
conv_layers = [l.name for l in model.layers if l.name.startswith("conv")]
conv_layers

In [ ]:
conv_layers = [l.name for l in model.layers if l.name.startswith("conv")]
conv_layers

# Define number of filters to visualize per layer and step size
num_filters = 8  # Number of filters to visualize per layer
filter_step = 4  # Step size to select filters

# Create a figure with subplots
fig, axes = plt.subplots(num_filters, len(conv_layers), figsize=(len(conv_layers) * 3, num_filters * 3))

# Iterate over each convolutional layer
for col, layer_name in enumerate(conv_layers):
    for row, filt in enumerate(range(0, num_filters * filter_step, filter_step)):
        loss, img = ex.visualize_filter(model, layer_name, filt, IMAGE_SIZE)
        
        # Plot the filter visualization
        axes[row, col].imshow(img)
        axes[row, col].axis("off")
        
        # Add labels for the first row and first column
        if row == 0:
            axes[row, col].set_title(layer_name, fontsize=30)
        if col == 0:
            axes[row, col].set_ylabel(f"Filter {filt}", fontsize=10)

# Adjust layout for better visibility
plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(4, 8, figsize=(10, 4))
axs = axs.flat

# Fill the picture with our saved filters
for filt in range(32):
    loss, img = ex.visualize_filter(model, "conv2d_4", filt, IMAGE_SIZE)
    axs[filt].imshow(img)
    axs[filt].axis("off")
plt.show()

In [ ]:
img_and_label = [x for x in val_ds.shuffle(100).take(100)]
x_val = [img.numpy() for img, _ in img_and_label]
y_val = [lab for _, lab in img_and_label]
y_val = np.concatenate(y_val, axis=0)
x_val = np.concatenate(x_val, axis=0)
choice = np.random.choice(x_val.shape[0], 30, replace=False)
x_val = x_val[choice]
y_val = y_val[choice]
y_val = [int(y[0]) for y in y_val]

In [ ]:
img_and_label = [x for x in train_ds.take(100)]
x_train = [img.numpy() for img, _ in img_and_label]

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format='retina'

from xplique.plots import plot_attributions
import xplique.attributions as xatt
import pandas as pd

In [ ]:
X = []
y = []
preds = []

for images, labels in val_ds:
    X.extend([img for img in images.numpy()])
    y.extend([lab for lab in labels.numpy()])
    preds.extend([pred for pred in model.predict(images, verbose=0)])

X = np.stack(X, axis=0)
y = np.stack(y, axis=0)
preds = np.stack(preds, axis=0)

In [ ]:
X_1 = X[y.ravel() == 1]
y_1 = y[y.ravel() == 1]
preds_1 = preds[y.ravel() == 1]

X_0 = X[y.ravel() == 0]
y_0 = y[y.ravel() == 0]
preds_0 = preds[y.ravel() == 0]

print(X_1.shape, X_0.shape)

X_1 = X_1[preds_1.ravel() > 0.9]
y_1 = y_1[preds_1.ravel() > 0.9]
preds_1 = preds_1[preds_1.ravel() > 0.9]

X_0 = X_0[preds_0.ravel() < 0.1]
y_0 = y_0[preds_0.ravel() < 0.1]
preds_0 = preds_0[preds_0.ravel() < 0.1]

print(X_1.shape, X_0.shape)

In [ ]:
idx = np.random.randint(0, X_1.shape[0], size=1000)

X_1 = X_1[idx]
y_1 = y_1[idx]
preds_1 = preds_1[idx]

idx = np.random.randint(0, X_0.shape[0], size=1000)

X_0 = X_0[idx]
y_0 = y_0[idx]
preds_0 = preds_0[idx]

print(X_1.shape, X_0.shape)

In [ ]:
def f(X):
    tmp = X.copy()
    return model(tmp)

label2class = {0: "aged", 1: "young"}

for i in range(X_1.shape[0] - 1):
    print(label2class[y_1[i][0]])

    im = np.expand_dims(X_1[i], axis=0)

    im_prob = model.predict([im])
    print(im_prob[0])
    
    # define a masker that is used to mask out partitions of the input image, this one uses a blurred background
    masker = shap.maskers.Image("inpaint_telea", X_1[0].shape)
    
    # By default the Partition explainer is used for all  partition explainer
    explainer = shap.Explainer(f, masker, output_names=["rejuvenation explanation"])
    
    # here we use 500 evaluations of the underlying model to estimate the SHAP values
    shap_values = explainer(X_1[i:i+1], max_evals=1000, batch_size=50)
    shap.image_plot(shap_values, labels=[im_prob[0][0]], true_labels=[label2class[y_1[i][0]]], vmax=0.0005)

In [ ]:
shap_model = tf.keras.models.clone_model(model)
model.layers[-1].activation = tf.keras.activations.linear

In [ ]:
def f(X):
    tmp = X.copy()
    return model(tmp)

def get_shap_explanations(model, X):
    
    # define a masker that is used to mask out partitions of the input image, this one uses a blurred background
    masker = shap.maskers.Image("inpaint_telea", X[0].shape)
    
    # By default the Partition explainer is used for all  partition explainer
    explainer = shap.Explainer(f, masker, output_names=["rejuvenation explanation"])
    
    # here we use 500 evaluations of the underlying model to estimate the SHAP values
    shap_values = explainer(X, max_evals=500, batch_size=50)

    return shap_values

shaps_1 = get_shap_explanations(model, X_1)
shaps_0 = get_shap_explanations(model, X_0)

In [ ]:
shaps_1.shape

In [ ]:
def get_nucleus_mask(img, res, tolerance=0.2):

    sigma = 0.1 / res

    smooth_img = ski.filters.gaussian(img, sigma=sigma)
    otsu_thres = ski.filters.threshold_otsu(smooth_img)
    lower_thres = otsu_thres * (1 - tolerance)
    upper_thres = otsu_thres * (1 + tolerance)

    # applies histerisis thresholding of TOLERANCE
    mask = ski.filters.apply_hysteresis_threshold(
        smooth_img, lower_thres, upper_thres
    )

    # fills holes
    mask = ski.morphology.binary_closing(mask, ski.morphology.ball(5))
    mask = binary_fill_holes(mask, ski.morphology.ball(5))

    return mask

def compute_exp_signal_by_distance(explanations, ims):

    data = []

    for exp_im, im in zip(explanations, ims):

        nuc_mask = get_nucleus_mask(im, RES)
        distance_map = distance_transform_edt(nuc_mask)  # Compute distance transform
    
        #plt.imshow(distance_map)
        #plt.show()
        #plt.imshow(nuc_mask)
        #plt.show()
        #plt.imshow(np.squeeze(im))
        #plt.show()
    
        # Flatten arrays
        distances = distance_map.flatten()
        exp_values = exp_im.flatten()
    
        # Bin distances and compute mean SHAP value per distance
        for d in np.unique(distances):
            mean_exp = exp_values[distances == d].mean()  # Compute mean SHAP for this distance
            data.append((int(d), mean_exp))

    df = pd.DataFrame(data, columns=["Distance", "EXP_Value"])
    return df

def get_local_entropy(images, disk_size):
    
    footprint = disk(disk_size)
    local_ent = []
    shannon_ent = []
    
    for im in images:

        #
        nuc_mask = get_nucleus_mask(im, RES)
        im, nuc_mask = np.squeeze(im), np.squeeze(nuc_mask)
        im = rescale_intensity(im, in_range=(im.min(), im.max()), out_range=(0, 255))
        im = im.astype(np.uint8)

        ent = entropy(im, footprint, mask=nuc_mask)
        local_ent.extend(ent.ravel())
        shannon_ent.append(shannon_entropy(im))

        """
        plt.imshow(im)
        plt.show()
        plt.imshow(ent)
        plt.show()
        """

    return local_ent, shannon_ent

In [ ]:
sns.color_palette("colorblind")
pal = utils.get_class_palette()
mic_pal = utils.get_microscopist_palette()

In [ ]:
# Convert SHAP values into a long-format DataFrame
df_shap = pd.DataFrame({
    "SHAP Value": np.concatenate([shaps_1.values.flatten(), shaps_0.values.flatten()]),
    "Condition": ["young"] * shaps_1.values.size + ["aged"] * shaps_0.values.size
})

In [ ]:
# Plot histogram using Seaborn
plt.figure(figsize=(4, 3))
sns.histplot(data=df_shap, x="SHAP Value", hue="Condition", bins=100, kde=True, 
             palette=pal, alpha=0.5, stat="density")

# Labels and formatting
plt.xlabel("SHAP Values")
plt.xlim((-0.003, 0.003))
plt.ylabel("Density")
plt.title("Distribution of SHAP Values")
plt.show()

In [ ]:
df_box = pd.DataFrame({
    "Shannon Entropy": np.concatenate([young_shannon_ent, aged_shannon_ent]),
    "Condition": ["young"] * len(young_shannon_ent) + ["aged"] * len(aged_shannon_ent)
})

plt.figure(figsize=(2, 2))
b = sns.boxplot(data=df_box, x="Condition", y="Shannon Entropy", palette=pal)
sns.stripplot(data=df_box, x="Condition", y="Shannon Entropy", 
              color="black", size=3, alpha=0.5, jitter=True)
plt.ylabel("Shannon Entropy")
plt.title("")
plt.grid(axis='y', linestyle="--", alpha=0.5)
plt.show()

In [ ]:
lab2class = {0.: "aged", 1.: "young"}

def plot_samples(samples, probs, labels):
    plt.figure(figsize=(15, 10))
    
    for i, img in enumerate(samples):
        
        ax = plt.subplot(1, len(samples), i + 1)
        label = lab2class[labels[i][0]]
        plt.imshow(tf.squeeze(img).numpy(), cmap='gray')
        plt.title(f'True: {label} \nProb: {probs[i][0]:.4f}')
        plt.axis('off')
    plt.show()

In [ ]:
from skimage.segmentation import felzenszwalb

def FS(x):
    x = felzenszwalb(x.numpy().astype('double'), 
                     scale=20, sigma=0.5, min_size=30)
    return tf.cast(x, tf.int32)

def get_explainers(model):
    return [
        xatt.Saliency(model),
        xatt.DeconvNet(model),
        xatt.GradientInput(model),
        xatt.GuidedBackprop(model),
        xatt.IntegratedGradients(model, steps=100, batch_size=BATCH_SIZE),
        xatt.SmoothGrad(model, nb_samples=100, batch_size=BATCH_SIZE),
        xatt.SquareGrad(model, nb_samples=100, batch_size=BATCH_SIZE),
        xatt.VarGrad(model, nb_samples=100, batch_size=BATCH_SIZE),
        xatt.GradCAM(model, conv_layer="conv2d_4"), # "conv2d_3"
        xatt.GradCAMPP(model, conv_layer="conv2d_4"),
        xatt.Occlusion(model, patch_size=10, patch_stride=5, batch_size=BATCH_SIZE),
        xatt.Rise(model, nb_samples=4000, batch_size=BATCH_SIZE),
        xatt.SobolAttributionMethod(model, batch_size=BATCH_SIZE),
        #xatt.KernelShap(model, nb_samples = 6000),
        xatt.KernelShap(model, nb_samples = 6000, map_to_interpret_space=FS),
        xatt.HsicAttributionMethod(model, nb_design=2000, batch_size=BATCH_SIZE)
    ]

explainers = get_explainers(model)

exp_dic = {}
n = 6

plot_samples(X_1[:n], preds_1[:n], y_1[:n])

for explainer in explainers:

    exp_name = explainer.__class__.__name__

    fake_y = np.array([1.] * len(y_1)).reshape([len(y_1), 1])

    explanations_1 = explainer(X_1, fake_y)

    # store the explanations to use the metrics
    exp_dic[exp_name] = explanations_1
    
    print(f"Method: {exp_name}")
    plot_attributions(explanations_1[:n], X_1[:n], img_size=4., cmap='jet', alpha=0.4,
                    cols=n, absolute_value=True, clip_percentile=0.5)
    plt.show()
    plot_attributions(explanations_1[:n], X_1[:n], img_size=4., cmap='seismic', alpha=0.4,
                    cols=n, absolute_value=False, clip_percentile=0.5)
    plt.show()
    print("\n")

In [ ]:
plot_samples(X_0[:n], preds_0[:n], y_0[:n])

for explainer in explainers:

    exp_name = explainer.__class__.__name__

    fake_y = np.array([1.] * len(y_0)).reshape([len(y_0), 1])

    explanations_0 = explainer(X_0, fake_y)

    # store the explanations to use the metrics
    exp_dic[exp_name] = explanations_0
    
    print(f"Method: {exp_name}")
    plot_attributions(explanations_0[:n], X_0[:n], img_size=4., cmap='jet', alpha=0.4,
                    cols=n, absolute_value=True, clip_percentile=0.5)
    plt.show()
    plot_attributions(explanations_0[:n], X_0[:n], img_size=4., cmap='seismic', alpha=0.4,
                    cols=n, absolute_value=False, clip_percentile=0.5)
    plt.show()
    print("\n")